# Fine-tuning: LoRA/QLoRA дообучение Mistral-7B

Датасет подготовлен в `01_data_preparation.ipynb` (250 примеров, style
transfer в стиль "лаконичный эксперт" через Groq API). Загружаем финальный
результат из Google Drive и переходим к обучению.

In [ ]:
import json

with open('/content/drive/MyDrive/final_style_transfer.jsonl', 'r') as f:
    records = [json.loads(line) for line in f]

print(f"Loaded {len(records)} examples")
print(records[0])

print(records[0]['response'])

250 examples downloaded
{'instruction': 'When did the first World war start?', 'context': '', 'original_response': 'July 28, 1914', 'response': 'July 28, 1914', 'category': 'open_qa'}
July 28, 1914


## Диагностика: поиск нестандартных символов

Проверяем все response на не-ASCII символы — находка ниже привела к
решению о точечной нормализации типографики (следующий раздел).

In [ ]:
import unicodedata

def find_unusual_chars(text):
    # isascii() cleanly separates "plain" characters from anything exotic —
    # covers both typographic symbols (smart quotes, special dashes) and
    # legitimate non-English content (diacritics, phonetic transcription)
    unusual = set()
    for ch in text:
        if ch.isascii():
            continue
        unusual.add(ch)
    return unusual

all_unusual = {}
for r in records:
    chars = find_unusual_chars(r['response'])
    for ch in chars:
        # unicodedata.name gives a human-readable label (e.g. "EM DASH")
        # instead of a raw character, making the output actually reviewable
        name = unicodedata.name(ch, f"U+{ord(ch):04X}")
        all_unusual[name] = all_unusual.get(name, 0) + 1

for name, count in sorted(all_unusual.items(), key=lambda x: -x[1]):
    print(f"{name}: {count} times")

LATIN SMALL LETTER O WITH ACUTE: 2 раз
GREEK SMALL LETTER BETA: 1 раз
LATIN SMALL LETTER E WITH GRAVE: 1 раз
LATIN SMALL LETTER E WITH DIAERESIS: 1 раз
LATIN SMALL LETTER C WITH ACUTE: 1 раз
LATIN SMALL LETTER N WITH ACUTE: 1 раз
LATIN SMALL LETTER ESH: 1 раз
LATIN SMALL LETTER ALPHA: 1 раз
MODIFIER LETTER LOW VERTICAL LINE: 1 раз
LATIN SMALL LETTER OPEN O: 1 раз
LATIN LETTER SMALL CAPITAL I: 1 раз
LATIN SMALL LETTER OPEN E: 1 раз
MODIFIER LETTER VERTICAL LINE: 1 раз
LATIN SMALL LETTER SCHWA: 1 раз
MODIFIER LETTER TRIANGULAR COLON: 1 раз
LATIN LETTER SMALL CAPITAL INVERTED R: 1 раз
LATIN SMALL LETTER A WITH RING ABOVE: 1 раз
LATIN SMALL LETTER I WITH DIAERESIS: 1 раз
LATIN SMALL LETTER A WITH CIRCUMFLEX: 1 раз
LATIN SMALL LETTER E WITH ACUTE: 1 раз
LATIN SMALL LETTER O WITH MACRON: 1 раз
LATIN SMALL LETTER I WITH ACUTE: 1 раз


## Точечная нормализация типографики

Заменяем только стилистические типографские символы (узкий неразрывный
пробел, "умные" кавычки, длинное/короткое тире), введённые моделью-
переписчиком — 254 случая по 7 категориям (см. диагностику выше).
Диакритика и фонетическая транскрипция (например, в ответе про Chardonnay)
намеренно не трогаются — это содержательная часть оригинального текста.

In [ ]:
import unicodedata

TYPOGRAPHIC_REPLACEMENTS = {
    '\u2011': '-',
    '\u202f': ' ',
    '\u2019': "'",
    '\u2018': "'",
    '\u201c': '"',
    '\u201d': '"',
    '\u2014': '-',
    '\u2013': '-',
}

def normalize_typography(text):
    for old, new in TYPOGRAPHIC_REPLACEMENTS.items():
        text = text.replace(old, new)
    return text

for r in records:
    r['response'] = normalize_typography(r['response'])
    r['original_response'] = normalize_typography(r['original_response'])


with open('final_style_transfer.jsonl', 'w') as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print("Normalized and saved")

Normalized and saved


In [ ]:
for r in records:
    if any(ch in r['response'] for ch in ['ə', 'ʃ', 'ː', 'ʁ', 'ʀ']):
        print(r['instruction'])
        print(r['response'][:200])

What is Chardonnay?
Chardonnay (UK /ˈʃɑːrdəneɪ/, US /ˌʃɑːrdənˈeɪ/, French [ʃaʁdɔnɛ]) is a green-skinned grape used for white wine. The grape is neutral, with wine flavors derived chiefly from terroir and oak. It is vinif


In [ ]:
# Ensure Google Drive is mounted: from google.colab import drive
# drive.mount('/content/drive')

with open('/content/final_style_transfer.jsonl', 'r') as f:
    clean_records = [json.loads(line) for line in f]

with open('/content/drive/MyDrive/final_style_transfer.jsonl', 'w') as f:
    for r in clean_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
with open('/content/drive/MyDrive/final_style_transfer.jsonl', 'r') as f:
    records = [json.loads(line) for line in f]

print(f"{len(records)} examples downloaded")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
250 examples downloaded


## Загрузка базовой модели в 4-bit (QLoRA)

Mistral-7B-Instruct-v0.2 — рекомендована в ТЗ, свободная лицензия Apache 2.0,
хорошо задокументированный QLoRA pipeline. Загружаем в 4-bit через
bitsandbytes, чтобы модель поместилась в 16GB VRAM бесплатного Colab T4
(в полной точности заняла бы ~14GB только под веса, без места под обучение).

In [ ]:
# Install necessary libraries for working with models, quantization, and datasets
!pip install -q transformers peft bitsandbytes accelerate datasets

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Identifier for the Mistral-7B-Instruct-v0.2 model from the Hugging Face repository
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",  # Uses Normalized Float 4: mathematically optimal for normally distributed neural network weights, minimizing quantization error
    bnb_4bit_compute_dtype = torch.float16, # float16, NOT bfloat16 — T4 (Turing architecture) lacks native bf16 tensor core acceleration; fp16 is the hardware-native choice here
    bnb_4bit_use_double_quant = True, # Nested quantization: compresses scaling factors a second time to save an extra ~0.4 bits per parameter
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Mistral lacks a default padding token; assigning eos_token prevents index out-of-bounds errors during batch tokenization.

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto",  # Automatically maps model layers across available GPUs and CPU memory to prevent initial OOM crashes.
)

print(f"Model loaded. Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded. Memory allocated: 8.97 GB


## Настройка логирования (Weights & Biases)

W&B логирует loss curve и метаданные обучения в реальном времени на
внешний сервис — данные остаются доступны по ссылке независимо от
состояния Colab-сессии (в отличие от простого print в консоль).

In [ ]:
!pip install -q wandb
import wandb
wandb.login()


In [ ]:
wandb.init(
    project = "laconic-expert-lora",
    name = "mistral-7b-qlora-run1",
    config={
        "model": "mistralai/Mistral-7B-Instruct-v0.2",
        "dataset_size": len(records),
        "method": "QLoRA",
    }
)


## Настройка LoRA-адаптера

Вместо дообучения всех 7B параметров модели (что физически не влезло бы
в память T4), замораживаем базовую модель и добавляем небольшие обучаемые
матрицы поверх attention-слоёв. Обучаем только их — в разы меньше памяти
и куда меньший риск переобучения на маленьком датасете (230 примеров).

Гиперпараметры:
- `r=16` — размер LoRA-матриц (rank); скромное значение, подходящее для
  небольшого датасета — большой r увеличивает риск переобучения
- `lora_alpha=32` — масштабирующий коэффициент, обычно берётся ×2 от r
- `target_modules` — слои attention-механизма (q/k/v/o projections),
  стандартный выбор для LoRA на моделях семейства Mistral/Llama
- `lora_dropout=0.05` — небольшая регуляризация, снижает переобучение
  при малом объёме данных

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias = "none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


## Форматирование данных под prompt template Mistral

Mistral обучен ожидать инструкции в специфическом формате `[INST]...[/INST]`.
Подача текста в произвольном формате ухудшила бы способность модели понять
границу между вопросом и ответом. Всё, что идёт после `[/INST]`, — это то,
чему модель учится отвечать.

In [ ]:
def format_example(example):
  context_part = f"\n\n{example['context']}" if example['context'] else ""
  # <s>/</s> — special begin/end-of-sequence tokens Mistral expects;
  # everything after [/INST] is the target the model learns to generate
  prompt = f"<s>[INST] {example['instruction']}{context_part} [/INST] {example['response']}</s>"
  return {"text": prompt}

formatted_data = [format_example(r) for r in records]
print(formatted_data[0]['text'])

<s>[INST] When did the first World war start? [/INST] July 28, 1914</s>


## Train/eval split: отложенные примеры для честной оценки

20 примеров исключаются из обучения и откладываются для Шага 3 (оценка).
Без этого разделения сравнение базовой и дообученной модели на тех же
данных, что видела модель при обучении, было бы нечестным — высокое
качество ответов могло бы означать не обобщение стиля, а простое
запоминание конкретных примеров (особенно вероятно при таком небольшом
датасете и нескольких эпохах обучения).

230 train / 20 eval — соответствует диапазону "10-20 примеров для оценки",
указанному в задании; данные перемешиваются с фиксированным random_state
для воспроизводимости разбиения.

In [ ]:
import random

random.seed(42)

random.shuffle(formatted_data)

# 20 examples held out entirely from training — used exclusively
# for the base-vs-finetuned comparison in the evaluation step

eval_holdout = formatted_data[:20]

train_data = formatted_data[20:]

print(f"Train: {len(train_data)}, Eval_holdout: {len(eval_holdout)}")

with open('/content/drive/MyDrive/eval_holdout.jsonl', 'w') as f:
    for ex in eval_holdout:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')



Train: 230, Eval_holdout: 20


## Токенизация с маскировкой паддинга

Модель учится предсказывать следующий токен (causal language modeling),
поэтому labels = input_ids. Но без маскировки паддинга модель получала бы
сигнал/штраф даже за предсказание пустых токенов-заполнителей в конце
коротких примеров (при max_length=512 многие instruction-response пары
значительно короче) — это создавало обучающий шум.

Это было реально обнаружено эмпирически: первый sanity-check без этой
маскировки давал хаотичный loss (14-17, выше уровня случайного угадывания
~10.4 для словаря Mistral) — после добавления -100 для паддинга и
исправления compute_dtype (см. выше) loss упал до адекватных 1.9-2.5
и стабильно снижался.

-100 — специальное значение, которое PyTorch автоматически игнорирует
при расчёте loss.

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)

def tokenize_function(examples):
  result = tokenizer(
      examples['text'],
      truncation=True,
      max_length=512,
      padding="max_length",
  )
  # mask padding tokens with -100 so they're excluded from the loss —
  # without this, the model is penalized for "predicting" empty filler tokens
  labels = []
  for ids in result["input_ids"]:
        label = [tok if tok != tokenizer.pad_token_id else -100 for tok in ids]
        labels.append(label)

  result["labels"] = labels

  return result

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(tokenized_train[0]["labels"][-20:])


## Обучение

Гиперпараметры:
- `num_train_epochs=3` — для маленького датасета (230 примеров) обычно
  достаточно 2-3 эпох; больше — риск переобучения (заучивание конкретных
  примеров вместо обобщения стиля)
- `per_device_train_batch_size=4` + `gradient_accumulation_steps=4` —
  эффективный размер батча 16 без превышения памяти T4 (16GB); реальный
  батч 16 не поместился бы одновременно в память вместе с активациями
- `learning_rate=2e-4` — стандартное значение для LoRA (выше, чем при
  полном fine-tuning, так как обучается намного меньше параметров)
- `fp16=True` — не `bf16`, так как T4 (архитектура Turing) не имеет
  нативной аппаратной поддержки bfloat16 (появилась только в Ampere и
  новее); также обязательно должно совпадать с compute_dtype в конфиге
  квантования — несовпадение (bf16 в одном месте, fp16 в другом) стало
  одной из причин аномального loss на раннем sanity-check

Перед полным прогоном на 3 эпохи проведена короткая проверка на 5 шагах
(`max_steps=5`), которая как раз и выявила описанные выше проблемы —
дешевле поймать баг за 2 минуты, чем после 20+ минут полного обучения.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = "./mistral-laconic-lora",
    num_train_epochs = 3,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    learning_rate = 2e-4,
    logging_steps= 5,
    save_strategy = "epoch",
    report_to = "wandb",
    run_name = "mistral-7b-qlora-run1",
    fp16 = True
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
)

trainer.train()

TrainOutput(global_step=45, training_loss=1.6399921205308703, metrics={'train_runtime': 1086.312, 'train_samples_per_second': 0.635, 'train_steps_per_second': 0.041, 'total_flos': 1.510121838477312e+16, 'train_loss': 1.6399921205308703, 'epoch': 3.0})

## Sanity-check перед полным обучением

Прежде чем запускать полный прогон на 3 эпохи (15-40 минут), проверяем
корректность всего пайплайна на 5 шагах. Дешевле поймать баг за 1-2 минуты
здесь, чем после долгого ожидания в основном прогоне.

Первый прогон (до исправлений ниже) дал аномальный, хаотичный loss
(14.9 → 16.8 → 13.1 → 16.6 → 7.8) — выше теоретического уровня случайного
угадывания (~10.4 для словаря Mistral). Диагностика показала две причины:
(1) `bnb_4bit_compute_dtype=torch.bfloat16` в конфиге квантования конфликтовал
с `fp16=True` в TrainingArguments — исправлено на согласованный `float16`
в обоих местах; (2) паддинг не был замаскирован в labels (см. функцию
токенизации выше — `-100` для pad-токенов) — без этого модель штрафовалась
за предсказание пустых заполнителей.

После обоих исправлений повторный sanity-check дал адекватный,
последовательно снижающийся loss (2.28 → 2.48 → 2.17 → 2.07 → 1.90) —
только после этого запущено полное обучение

In [ ]:
from transformers import TrainingArguments, Trainer

sanity_args = TrainingArguments(
    output_dir="./sanity-check",
    max_steps=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    fp16=True,
    report_to="none",
)

sanity_trainer = Trainer(model=model, args=sanity_args, train_dataset=tokenized_train)
sanity_trainer.train()

TrainOutput(global_step=5, training_loss=2.181000995635986, metrics={'train_runtime': 121.8136, 'train_samples_per_second': 0.657, 'train_steps_per_second': 0.041, 'total_flos': 1750865899683840.0, 'train_loss': 2.181000995635986, 'epoch': 0.3448275862068966})

## Сохранение LoRA-адаптера

Сохраняем именно адаптер  — компактный
набор весов, который применяется поверх оригинальной, неизменной базовой
модели. Копируем в Google Drive, чтобы адаптер пережил закрытие сессии
Colab (в отличие от файлов в /content/).

In [ ]:
model.save_pretrained("./mistral-laconic-lora-adapter")
tokenizer.save_pretrained("./mistral-laconic-lora-adapter")

import shutil
shutil.copytree("./mistral-laconic-lora-adapter", "/content/drive/MyDrive/mistral-laconic-lora-adapter")

print("Adapter saved in Drive")

import os
print(os.path.exists("/content/drive/MyDrive/mistral-laconic-lora-adapter/adapter_model.safetensors"))

True
